In [1]:
import os
import csv
import pandas as pd
from datetime import datetime

# Project paths
DATASET_DIR = "../dataset"

CHAT_HISTORY_FILE = os.path.join(DATASET_DIR, "chat_history.csv")
SAMPLE_QUESTIONS_FILE = os.path.join(DATASET_DIR, "sample_questions.csv")
RESPONSES_FILE = os.path.join(DATASET_DIR, "responses.csv")
LOG_FILE = os.path.join(DATASET_DIR, "chatbot_logs.txt")

print("Week 9 - Education AI Chatbot")
print("Notebook started successfully!")
print("Current time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Week 9 - Education AI Chatbot
Notebook started successfully!
Current time: 2026-10-28 14:45:39


In [2]:
# Load the project datasets

if os.path.exists(SAMPLE_QUESTIONS_FILE):
    questions_df = pd.read_csv(SAMPLE_QUESTIONS_FILE)
    print("Sample Questions Dataset")
    display(questions_df.head())
    print("Total sample questions:", len(questions_df))
else:
    print("Sample questions file not found:", SAMPLE_QUESTIONS_FILE)

if os.path.exists(RESPONSES_FILE):
    responses_df = pd.read_csv(RESPONSES_FILE)
    print("\nChatbot Responses Dataset")
    display(responses_df.head())
    print("Total saved response rows:", len(responses_df))
else:
    print("Responses file not found:", RESPONSES_FILE)

if os.path.exists(CHAT_HISTORY_FILE):
    history_df = pd.read_csv(CHAT_HISTORY_FILE)
    print("\nInteractive Chat History")
    display(history_df.head())
    print("Total chat history rows:", len(history_df))
else:
    print("Chat history file not found:", CHAT_HISTORY_FILE)

Sample Questions Dataset


,question_id,category,question
0,1,Conceptual,What is photosynthesis? Explain in simple Engl...
1,2,Conceptual,What is artificial intelligence? Give one exam...
2,3,Conceptual,Explain machine learning with a real-life exam...
3,4,Conceptual,What is a database and why is it used?
4,5,Conceptual,Explain the difference between hardware and so...


Total sample questions: 20

Chatbot Responses Dataset


,timestamp,question_id,category,question,model,response,response_length_words
0,2026-10-28 14:20:29,1,Conceptual,What is photosynthesis? Explain in simple Engl...,gemma3:1b,"Okay, let's break down photosynthesis! \n\n**W...",538
1,2026-10-28 14:20:35,2,Conceptual,What is artificial intelligence? Give one exam...,gemma3:1b,"Okay, let’s break down what Artificial Intelli...",436
2,2026-10-28 14:20:45,3,Conceptual,Explain machine learning with a real-life exam...,gemma3:1b,"Okay, let's break down machine learning. It’s ...",652
3,2026-10-28 14:20:54,4,Conceptual,What is a database and why is it used?,gemma3:1b,"Okay, let's break down databases!\n\n**What is...",563
4,2026-10-28 14:21:03,5,Conceptual,Explain the difference between hardware and so...,gemma3:1b,"Okay, let’s break down the difference between ...",518


Total saved response rows: 60

Interactive Chat History


,timestamp,user_message,chatbot_response
0,2026-10-28 13:35:20,My name is Kaviya.,"It’s nice to meet you, Kaviya! 😊 \n\nIs there ..."
1,2026-10-28 13:38:10,My name is Kaviya.,"It's nice to meet you, Kaviya! 😊 \n\nIs there ..."
2,2026-10-28 13:38:25,What is my name?,"As a large language model, I don't have access..."
3,2026-10-28 13:49:45,My name is Kaviya.,"It’s nice to meet you, Kaviya! It’s good to kn..."
4,2026-10-28 13:49:59,What is my name?,Your name is Kaviya. 😊


Total chat history rows: 8


In [3]:
# Dataset summary and response analysis

print("===== DATASET SUMMARY =====")

if "questions_df" in globals():
    print("Total sample questions:", len(questions_df))

    if "category" in questions_df.columns:
        print("\nQuestions by category:")
        display(
            questions_df["category"]
            .value_counts()
            .rename_axis("Category")
            .reset_index(name="Count")
        )

if "responses_df" in globals():
    print("\nTotal saved responses:", len(responses_df))

    if "question_id" in responses_df.columns:
        print("\nResponses by type:")
        display(
            responses_df["question_id"]
            .astype(str)
            .str.extract(r"-(original|improved)$", expand=False)
            .fillna("sample")
            .value_counts()
            .rename_axis("Response Type")
            .reset_index(name="Count")
        )

    if "response_length_words" in responses_df.columns:
        print("\nResponse length statistics:")
        display(
            responses_df["response_length_words"]
            .describe()
            .to_frame(name="Word Count")
        )

===== DATASET SUMMARY =====
Total sample questions: 20

Questions by category:


,Category,Count
0,Conceptual,5
1,Reasoning,5
2,Coding,5
3,General,5



Total saved responses: 60

Responses by type:


,Response Type,Count
0,sample,20
1,original,20
2,improved,20



Response length statistics:


,Word Count
count,60.000000
mean,435.100000
std,204.169313
min,86.000000
25%,235.500000
50%,490.000000
75%,563.250000
max,944.000000


In [4]:
# Compare original and improved prompt responses

if "responses_df" in globals() and "question_id" in responses_df.columns:

    comparison_df = responses_df[
        responses_df["question_id"]
        .astype(str)
        .str.endswith(("-original", "-improved"))
    ].copy()

    comparison_df["prompt_type"] = comparison_df["question_id"].apply(
        lambda value: (
            "Original" if str(value).endswith("-original")
            else "Improved"
        )
    )

    comparison_df["base_question_id"] = (
        comparison_df["question_id"]
        .astype(str)
        .str.replace(r"-(original|improved)$", "", regex=True)
    )

    print("Prompt comparison response counts:")
    display(
        comparison_df["prompt_type"]
        .value_counts()
        .rename_axis("Prompt Type")
        .reset_index(name="Responses")
    )

    if "response_length_words" in comparison_df.columns:
        print("\nAverage response length by prompt:")
        display(
            comparison_df.groupby("prompt_type")[
                "response_length_words"
            ].mean().round(2).reset_index()
        )

    print("\nExample: Original and Improved answers for Question 1")

    for prompt_type in ["Original", "Improved"]:
        example = comparison_df[
            (comparison_df["base_question_id"] == "1")
            & (comparison_df["prompt_type"] == prompt_type)
        ]

        if not example.empty:
            print(f"\n--- {prompt_type} Prompt ---")
            print(example.iloc[0]["response"])

else:
    print("Run the dataset loading cell first.")

Prompt comparison response counts:


,Prompt Type,Responses
0,Original,20
1,Improved,20



Average response length by prompt:


,prompt_type,response_length_words
0,Improved,574.10
1,Original,199.55



Example: Original and Improved answers for Question 1

--- Original Prompt ---
Okay, let’s talk about photosynthesis!

**Basically, photosynthesis is how plants make their own food.**

Think of it like this: plants are like little chefs, and they need ingredients to cook.  They get those ingredients from the air and from the ground.

Here’s how it works:

1. **They take in sunlight:**  Plants have special stuff called chlorophyll (that's what makes them green!) that grabs the sunlight like a solar panel.

2. **They take in water:**  They get this from their roots.

3. **They take in carbon dioxide:**  This is a gas that we breathe out – it's in the air.

4. **They mix it all together:**  Using the sunlight's energy, they magically turn water and carbon dioxide into sugar (their food!) and oxygen. 

**The sugar helps them grow big and strong.**

And as a bonus, they release the oxygen into the air for us to breathe!

**So, photosynthesis is a really important process that keeps plants 

In [5]:
# Final Results and Conclusion

print("=" * 50)
print("WEEK 9 - EDUCATION AI CHATBOT")
print("FINAL RESULTS AND CONCLUSION")
print("=" * 50)

if "questions_df" in globals():
    total_questions = len(questions_df)
    print(f"\nTotal sample questions: {total_questions}")

if "responses_df" in globals():
    print(f"Total saved response rows: {len(responses_df)}")

    if "question_id" in responses_df.columns:
        ids = responses_df["question_id"].astype(str)

        original_count = ids.str.endswith("-original").sum()
        improved_count = ids.str.endswith("-improved").sum()

        print(f"Original prompt responses: {original_count}")
        print(f"Improved prompt responses: {improved_count}")

print("\nKey Observations:")
print("1. The chatbot uses the Gemma 3 1B model.")
print("2. Sample questions cover multiple educational categories.")
print("3. Prompt comparison tests two different instruction styles.")
print("4. Response length alone does not establish answer quality.")
print("5. Responses should be checked for accuracy, clarity, and relevance.")

print("\nConclusion:")
print(
    "Prompt engineering can guide the structure and clarity of "
    "educational chatbot responses. Comparing saved outputs helps "
    "evaluate which prompt is more useful for different questions. "
    "The best prompt should be selected based on actual answer "
    "quality and correctness."
)

WEEK 9 - EDUCATION AI CHATBOT
FINAL RESULTS AND CONCLUSION

Total sample questions: 20
Total saved response rows: 60
Original prompt responses: 20
Improved prompt responses: 20

Key Observations:
1. The chatbot uses the Gemma 3 1B model.
2. Sample questions cover multiple educational categories.
3. Prompt comparison tests two different instruction styles.
4. Response length alone does not establish answer quality.
5. Responses should be checked for accuracy, clarity, and relevance.

Conclusion:
Prompt engineering can guide the structure and clarity of educational chatbot responses. Comparing saved outputs helps evaluate which prompt is more useful for different questions. The best prompt should be selected based on actual answer quality and correctness.
